# Modelos avançados no Kaggle

Este notebook **não roda no ambiente local** — foi gerado aqui, mas é
feito para ser subido e executado no Kaggle, onde PyTorch + GPU já vêm
prontos (ver `reports/tables/modelos_avancados_smoke_test.md` para o
porquê de não dar para rodar isso localmente).

Segue a mesma organização dos outros notebooks deste repositório: o código
fica no pacote `src/churn_telecom/`, e este notebook só clona o repositório,
instala o pacote e orquestra os experimentos.

**No Kaggle, antes de rodar:** ative *Settings → Internet → On* (o
notebook clona o repositório, baixa o CSV e instala os pacotes) e
*Accelerator → GPU*. Depois use **"Save Version" → "Save & Run All
(Commit)"**, que executa num kernel gerenciado em segundo plano.


In [ ]:
#@title Setup (clona o repo e instala o pacote)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jpbezerra/redes-neurais.git"   # ajuste se necessario
BRANCH   = "feat/churn"   # ajuste aqui se a branch de trabalho mudar
REPO_DIR = "redes-neurais"
SUBPATH  = "projeto/churn-telecom"

def _sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=False)

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    base = Path("/content") if IN_COLAB else Path("/kaggle/temp")  # fora de /kaggle/working: o clone nao entra no Output
    root = base / REPO_DIR
    if root.exists():
        _sh(f"cd {root} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
    else:
        _sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {root}")
    PROJECT_ROOT = root / SUBPATH
    _sh(f"pip install -q -e {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd().parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# No Kaggle, tudo que for gerado vai para /kaggle/working/saida (so isso aparece
# no Output, em vez do repositorio inteiro).
OUT_DIR = Path("/kaggle/working/saida") if IN_KAGGLE else PROJECT_ROOT

DATA_DIR    = PROJECT_ROOT / "data"
RESULTS_DIR = OUT_DIR / "results"
TABLES_DIR  = OUT_DIR / "reports" / "tables"
for d in (DATA_DIR, RESULTS_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR :", RESULTS_DIR)


## Instalação dos modelos avançados

O Kaggle já vem com PyTorch e a maioria das libs de ML — só falta instalar
os 5 modelos em si.

In [2]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00


## Preparação dos dados

`load_raw` baixa o CSV automaticamente se não existir em `DATA_DIR`
(mesma conveniência do mini-projeto 2) — não é preciso subir nada como
Kaggle Dataset separado, o clone do repositório já basta.

In [ ]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata
from churn_telecom.ensemble import corrigir_prior, salvar_predicoes, media_probabilidades

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

# Versao SEM linhas duplicadas do treino (mesmas linhas originais, sem oversampling).
# TabPFN/Mitra fazem in-context learning: linhas repetidas viram memorizacao e
# distorcem a calibracao, entao eles tambem sao treinados nesta versao.
df_train_nat = df.iloc[split.train_no_oversample_idx]
X_train_nat, _ = prep.transform(df_train_nat)
y_train_nat = prep.transform_target(df_train_nat)
PRIOR_NAT = float(y_train_nat.mean())

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"X_train_nat (sem duplicatas): {X_train_nat.shape} | prior de churn: {PRIOR_NAT:.3f}")


## Execução isolada por modelo

Cada modelo roda dentro de `rodar(nome, fn)`: se um falhar (pacote,
download de pesos, memória...), o erro é registrado em `FALHAS` e o
notebook **continua** para os próximos — um "Save & Run All" nunca mais
morre no meio por causa de um único modelo. As falhas são salvas em
`reports/tables/modelos_avancados_falhas_kaggle.csv` no final.

In [ ]:
import traceback

FALHAS = {}

def guardar_preds(model_id, prever, corrigir=None):
    # Salva as probabilidades de validacao/teste do modelo (para ensembles
    # e consolidacao posterior sem retreinar). `corrigir` ajusta o prior.
    pv, pt = prever(X_val), prever(X_test)
    if corrigir is not None:
        pv, pt = corrigir(pv), corrigir(pt)
    salvar_predicoes(RESULTS_DIR, model_id, pv, pt)
    return pv, pt

def rodar(nome, fn):
    try:
        fn()
        print(f"[ok] {nome}")
    except Exception as e:
        FALHAS[nome] = f"{type(e).__name__}: {e}"
        print(f"[FALHOU] {nome}: {type(e).__name__}: {e}")
        traceback.print_exc()


## STab (implementação própria — ver `src/churn_telecom/models/stab.py`)

Não existe um pacote público legítimo chamado "STab" (colisão de nome no
PyPI — ver smoke test). Esta é uma implementação própria e enxuta de
Transformer para dados tabulares, cobrindo o espaço de hiperparâmetros
pedido no enunciado.

In [ ]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

def experimento_stab():
    hp = STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1,
                         lr=1e-3, weight_decay=1e-5, batch_size=64, max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_stab(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    metricas = full_report(y_test, prever_stab(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "stab_churn", framework="PyTorch (STab — implementacao propria)",
             hyperparameters=hp.__dict__, metrics=metricas,
             architecture={"dim": hp.dim, "depth": hp.depth, "heads": hp.heads},
             history=historico, notes="STab (implementacao propria).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))
    guardar_preds("stab_churn", lambda X: prever_stab(modelo, X, device=device))

rodar("stab", experimento_stab)


## TabPFN v2

A versão é fixada em **v2** (a do enunciado). Versões novas do pacote usam
por padrão checkpoints v2.5+, que exigem aceitar licença e um
`TABPFN_TOKEN` — inviável num "Save & Run All". Os pesos v2 baixam direto.

In [ ]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

def experimento_tabpfn(model_id, X_ctx, y_ctx, prior_ctx, n_estimators=8):
    hp = TabPFNHyperparams(device=device, n_estimators=n_estimators, versao="v2", seed=SEED)
    modelo = treinar_tabpfn(X_ctx, y_ctx, hp)
    # prior_ctx != None -> modelo aprendeu a proporcao natural de churn; leva as
    # probabilidades para a escala "treino balanceado" (limiar 0.5 comparavel; KS inalterado)
    corr = None if prior_ctx is None else (lambda p: corrigir_prior(p, prior_ctx))
    prever = lambda X: prever_tabpfn(modelo, X)
    p_test = prever(X_test) if corr is None else corr(prever(X_test))
    metricas = full_report(y_test, p_test)
    print(model_id, metricas)
    save_run(RESULTS_DIR, model_id, framework="TabPFN v2",
             hyperparameters={**hp.__dict__, "n_linhas_contexto": len(X_ctx), "prior_corrigido": prior_ctx},
             metrics=metricas, notes="TabPFN v2 (checkpoint v2, sem token).",
             save_model_fn=lambda d: None)  # pesos sao os do pacote (45 MB+): nao versionamos
    guardar_preds(model_id, prever, corr)

# (a) contexto com linhas duplicadas pelo oversampling (como na execucao anterior)
rodar("tabpfn", lambda: experimento_tabpfn("tabpfn_churn", X_train, y_train, None))
# (b) contexto so com linhas originais (sem duplicatas) + correcao de prior
rodar("tabpfn_sem_duplicatas", lambda: experimento_tabpfn("tabpfn_churn_sem_duplicatas", X_train_nat, y_train_nat, PRIOR_NAT, n_estimators=16))


## KAN (pykan)

O `.fit()` nativo do pykan deu **NaN** logo no primeiro passo na execução
anterior (KS 0.04, AUROC nan). O modelo do pykan agora é treinado com loop
próprio (mini-batch, Adam, gradient clipping, early stopping); se divergir,
levanta erro explícito em vez de gravar métrica inválida.

In [ ]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

def experimento_kan():
    hp = KANHyperparams(hidden_width=8, grid=3, k=3, lr=3e-3, batch_size=128,
                        max_epochs=100, patience=8, seed=SEED)
    modelo, historico = treinar_kan(X_train, y_train, X_val, y_val, hp, device=device, verbose=True)
    metricas = full_report(y_test, prever_kan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "kan_churn", framework="pykan (KAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "grid": hp.grid, "k": hp.k},
             history=pd.DataFrame(historico), notes="KAN (pykan, loop de treino proprio).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))
    guardar_preds("kan_churn", lambda X: prever_kan(modelo, X, device=device))

rodar("kan", experimento_kan)


## TabKAN (ChebyshevKAN do pacote `tabkan`)

O pacote `tabkan` expõe módulos PyTorch (`ChebyshevKAN`, `FourierKAN`, ...),
não um classificador sklearn-like — o wrapper usa `ChebyshevKAN` com
treino Adam + early stopping na validação, igual ao STab.

In [ ]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

def experimento_tabkan():
    hp = TabKANHyperparams(hidden_width=10, degree=3, lr=1e-3, batch_size=64,
                           max_epochs=200, patience=10, seed=SEED)
    modelo, historico = treinar_tabkan(X_train, y_train, X_val, y_val, hp, device=device)
    metricas = full_report(y_test, prever_tabkan(modelo, X_test, device=device))
    print(metricas)
    save_run(RESULTS_DIR, "tabkan_churn", framework="tabkan (ChebyshevKAN)", hyperparameters=hp.__dict__,
             metrics=metricas, architecture={"hidden_width": hp.hidden_width, "degree": hp.degree},
             history=historico, notes="TabKAN (ChebyshevKAN).",
             save_model_fn=lambda d: torch.save(modelo.state_dict(), d / "model.pt"))
    guardar_preds("tabkan_churn", lambda X: prever_tabkan(modelo, X, device=device))

rodar("tabkan", experimento_tabkan)


## Mitra (via AutoGluon)

In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra, salvar_mitra

def experimento_mitra(model_id, X_ctx, y_ctx, prior_ctx):
    hp = MitraHyperparams(time_limit_segundos=600, num_gpus=1 if device == "cuda" else 0, seed=SEED)
    preditor = treinar_mitra(X_ctx, y_ctx, cols, hp, path=f"/kaggle/temp/ag_{model_id}")
    corr = None if prior_ctx is None else (lambda p: corrigir_prior(p, prior_ctx))
    prever = lambda X: prever_mitra(preditor, X, cols)
    p_test = prever(X_test) if corr is None else corr(prever(X_test))
    metricas = full_report(y_test, p_test)
    print(model_id, metricas)
    # o preditor do AutoGluon (~300 MB) vai no zip, mas NAO e enviado ao GitHub (limite de 100 MB/arquivo)
    save_run(RESULTS_DIR, model_id, framework="AutoGluon (extra [mitra])",
             hyperparameters={**hp.__dict__, "n_linhas_contexto": len(X_ctx), "prior_corrigido": prior_ctx},
             metrics=metricas, notes="Mitra.",
             save_model_fn=lambda d: salvar_mitra(preditor, d / "autogluon_predictor"))
    guardar_preds(model_id, prever, corr)

rodar("mitra", lambda: experimento_mitra("mitra_churn", X_train, y_train, None))
rodar("mitra_sem_duplicatas", lambda: experimento_mitra("mitra_churn_sem_duplicatas", X_train_nat, y_train_nat, PRIOR_NAT))


## Baselines refeitos aqui (entram no ensemble)

GB, XGBoost e MLP rodam em segundos; refazê-los no Kaggle garante que suas
probabilidades de validação/teste estejam salvas (`predicoes.npz`) com a
mesma partição dos modelos avançados.

In [ ]:
from churn_telecom.models.boosting import (GradientBoostingHyperparams, treinar_gradient_boosting,
                                           XGBoostHyperparams, treinar_xgboost)
from churn_telecom.models.mlp import MLPHyperparams, treinar_mlp

def experimento_baselines():
    gb = treinar_gradient_boosting(X_train, y_train, GradientBoostingHyperparams(seed=SEED))
    xgbm = treinar_xgboost(X_train, y_train, XGBoostHyperparams(seed=SEED))
    mlp, hist_mlp = treinar_mlp(X_train, y_train, X_val, y_val, MLPHyperparams(seed=SEED))
    for mid, modelo, hp, hist in [("gb_churn_kaggle", gb, GradientBoostingHyperparams(seed=SEED), None),
                                  ("xgb_churn_kaggle", xgbm, XGBoostHyperparams(seed=SEED), None),
                                  ("mlp_churn_kaggle", mlp, MLPHyperparams(seed=SEED), hist_mlp)]:
        prever = lambda X, m=modelo: m.predict_proba(X)[:, 1]
        metricas = full_report(y_test, prever(X_test))
        print(mid, {k: round(v, 4) for k, v in metricas.items() if k in ("ks", "auroc", "recall", "precision")})
        save_run(RESULTS_DIR, mid, framework="scikit-learn/xgboost", hyperparameters=hp.__dict__,
                 metrics=metricas, history=hist, notes="Baseline refeito no Kaggle (para ensemble).",
                 save_model_fn=lambda d, m=modelo: joblib.dump(m, d / "model.joblib"))
        guardar_preds(mid, prever)

rodar("baselines", experimento_baselines)


## Avaliação robusta (várias partições)

O conjunto de teste tem ~1760 linhas (~470 churners), então o KS de **uma**
partição tem erro-padrão ≈ 0.02 — maior que a diferença entre quase todos os
modelos. Aqui o split de 3 etapas é repetido com 5 sementes (mesmo
procedimento, sem vazamento) e reportamos **média ± desvio** de KS/AUROC por
modelo, mais a média dos modelos (ensemble). É a comparação que sustenta
dizer qual modelo é melhor.

In [ ]:
from churn_telecom.robustez import avaliar_multiplas_particoes, resumir
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

def f_gb(Xtr, ytr, Xva, yva, sd):
    m = treinar_gradient_boosting(Xtr, ytr, GradientBoostingHyperparams(seed=sd))
    return lambda X: m.predict_proba(X)[:, 1]

def f_xgb(Xtr, ytr, Xva, yva, sd):
    m = treinar_xgboost(Xtr, ytr, XGBoostHyperparams(seed=sd))
    return lambda X: m.predict_proba(X)[:, 1]

def f_mlp(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_mlp(Xtr, ytr, Xva, yva, MLPHyperparams(seed=sd))
    return lambda X: m.predict_proba(X)[:, 1]

def f_stab(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_stab(Xtr, ytr, Xva, yva, STabHyperparams(seed=sd), device=device)
    return lambda X: prever_stab(m, X, device=device)

def f_tabkan(Xtr, ytr, Xva, yva, sd):
    m, _ = treinar_tabkan(Xtr, ytr, Xva, yva, TabKANHyperparams(seed=sd), device=device, verbose=False)
    return lambda X: prever_tabkan(m, X, device=device)

def avaliacao_robusta():
    global ROBUSTO
    longo = avaliar_multiplas_particoes(df, {"gb": f_gb, "xgb": f_xgb, "mlp": f_mlp,
                                             "stab": f_stab, "tabkan": f_tabkan}, seeds=range(5))
    longo.to_csv(TABLES_DIR / "avaliacao_robusta_por_particao_kaggle.csv", index=False)
    ROBUSTO = resumir(longo)
    ROBUSTO.to_csv(TABLES_DIR / "avaliacao_robusta_resumo_kaggle.csv", index=False)
    print(ROBUSTO.round(4).to_string())

rodar("avaliacao_robusta", avaliacao_robusta)


## Comparação final — todos os modelos do projeto

Funciona mesmo que algum modelo acima tenha falhado.

In [10]:
pd.DataFrame([{"modelo": k, "erro": v} for k, v in FALHAS.items()],
             columns=["modelo", "erro"]).to_csv(TABLES_DIR / "modelos_avancados_falhas_kaggle.csv", index=False)
print("Falhas:", FALHAS or "nenhuma")

resumo = load_all_metadata(RESULTS_DIR)
cols_mostrar = ["model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
cols_existentes = [c for c in cols_mostrar if c in resumo.columns]
tabela = resumo[cols_existentes].sort_values("metrics.ks", ascending=False).reset_index(drop=True)
tabela


Falhas: nenhuma


,model_id,metrics.ks,metrics.auroc,metrics.recall,metrics.precision
0,gb_churn_baseline_oversample,0.524183,0.831954,0.743590,0.532110
1,stab_churn,0.522374,0.839526,0.752137,0.524590
2,xgb_churn_baseline_oversample,0.519855,0.831416,0.752137,0.528529
3,gb_churn_features_charges_per_tenure,0.516083,0.831512,0.747863,0.529501
4,gb_churn_baseline_classweight,0.512136,0.831246,0.758547,0.512266
5,mlp_churn_baseline_oversample,0.511046,0.829421,0.767094,0.493132
6,mlp_churn_optuna_leva1,0.510758,0.816685,0.724359,0.544141
7,gb_churn_optuna_leva1,0.510391,0.835575,0.756410,0.513043
8,xgb_churn_baseline_scaleposweight,0.509767,0.833499,0.767094,0.515805
9,tabkan_churn,0.503431,0.834274,0.752137,0.511628


## Ensemble dos modelos desta execução

Usa as probabilidades salvas (`predicoes.npz`). Dois ensembles: média de
**todos** os modelos e média dos **3 melhores na validação** (a seleção usa
só a validação — o teste não participa da escolha).

In [ ]:
def construir_ensembles():
    ids = sorted(d.name for d in RESULTS_DIR.iterdir() if (d / "predicoes.npz").exists())
    P = {i: np.load(RESULTS_DIR / i / "predicoes.npz") for i in ids}
    ks_val = {i: full_report(y_val, P[i]["p_val"])["ks"] for i in ids}
    ordem = sorted(ids, key=lambda i: -ks_val[i])
    print("KS de validacao:", {i: round(ks_val[i], 4) for i in ordem})
    for nome, membros in [("ensemble_todos_churn", ids), ("ensemble_top3_val_churn", ordem[:3])]:
        pv = media_probabilidades([P[m]["p_val"] for m in membros])
        pt = media_probabilidades([P[m]["p_test"] for m in membros])
        metricas = full_report(y_test, pt)
        print(nome, membros, {k: round(v, 4) for k, v in metricas.items() if k in ("ks", "auroc", "recall", "precision")})
        save_run(RESULTS_DIR, nome, framework="media de probabilidades", hyperparameters={"membros": membros},
                 metrics=metricas, notes="Ensemble por media simples das probabilidades.", save_model_fn=lambda d: None)
        salvar_predicoes(RESULTS_DIR, nome, pv, pt)

rodar("ensemble", construir_ensembles)


## Salvando os resultados (sem baixar o repositório inteiro)

Esta célula faz duas coisas:

1. Gera `results_churn.zip` só com `results/` + tabelas (poucos MB) — no
   Output do Kaggle aparece só isso, não o repositório clonado.
2. **Se existir o segredo `GITHUB_TOKEN`** (Kaggle → *Add-ons → Secrets*;
   token do GitHub com permissão *Contents: Read and write* neste repo),
   faz push dos resultados para a branch `results/churn-kaggle`. Localmente:
   `git fetch origin results/churn-kaggle` e
   `git checkout origin/results/churn-kaggle -- projeto/churn-telecom/results projeto/churn-telecom/reports/tables`.

Sem o segredo, só o zip é gerado.

In [ ]:
import shutil

RESULTS_BRANCH = "results/churn-kaggle"
LIMITE_MB = 20   # GitHub rejeita >100 MB por arquivo (o preditor do Mitra tem ~290 MB)

# falhas dos modelos ficam registradas junto das tabelas
pd.DataFrame([{"modelo": k, "erro": v} for k, v in FALHAS.items()],
             columns=["modelo", "erro"]).to_csv(TABLES_DIR / "modelos_avancados_falhas_kaggle.csv", index=False)
print("Falhas:", FALHAS or "nenhuma")

if IN_KAGGLE:
    # 1) zip COMPLETO (inclui pesos grandes) no Output
    zip_path = shutil.make_archive("/kaggle/working/results_churn", "zip", OUT_DIR)
    print("Zip gerado:", zip_path)

    # 2) push opcional: so arquivos pequenos (metadata, history, predicoes, tabelas, pesos leves)
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        token = None

    if not token:
        print("Sem segredo GITHUB_TOKEN: pulando o push (baixe o zip pelo Output).")
    else:
        try:
            ignorar = lambda pasta, nomes: []
            def copiar_pequenos(origem, destino, pulados):
                for f in Path(origem).rglob("*"):
                    if f.is_file():
                        if f.stat().st_size > LIMITE_MB * 1024 * 1024:
                            pulados.append(f"{f.relative_to(origem)} ({f.stat().st_size / 1e6:.0f} MB)")
                            continue
                        alvo = Path(destino) / f.relative_to(origem)
                        alvo.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(f, alvo)

            pulados = []
            copiar_pequenos(RESULTS_DIR, PROJECT_ROOT / "results", pulados)
            copiar_pequenos(TABLES_DIR, PROJECT_ROOT / "reports" / "tables", pulados)
            if pulados:
                print(f"Arquivos > {LIMITE_MB} MB NAO enviados ao GitHub (estao no zip):")
                for x in pulados:
                    print("  -", x)

            def git(*args):
                r = subprocess.run(["git", "-C", str(root), "-c", "user.name=kaggle-run",
                                    "-c", "user.email=kaggle-run@users.noreply.github.com", *args],
                                   capture_output=True, text=True)
                saida = (r.stdout + r.stderr).replace(token, "***")
                if r.returncode != 0:
                    raise RuntimeError(f"git {args[0]} falhou: {saida}")
                return saida

            git("checkout", "-B", RESULTS_BRANCH)
            git("add", "-f", f"{SUBPATH}/results", f"{SUBPATH}/reports/tables")
            git("commit", "-m", "results: saidas do notebook de modelos avancados (Kaggle)")
            url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
            git("push", "--force", url, f"HEAD:refs/heads/{RESULTS_BRANCH}")
            print(f"Resultados enviados para a branch {RESULTS_BRANCH}.")
        except Exception as e:
            # o push e conveniencia: se falhar, o zip acima continua valendo
            print("Push falhou (o zip continua valido):", str(e)[:600])
